In [0]:
########################## Country cleanup
countries_table = 'pollution_per_country_pipeline_project.bronze.countries'
countries = spark.read.table(countries_table)

# remove dupes
print(f"Count before de-duplication : {countries.count()}")
countries_unique = countries.drop_duplicates(['id'])
print(f"Count after de-duplication : {countries_unique.count()}")

# remove region rows
only_countries = countries_unique.filter(countries_unique['region']['value'] != 'Aggregates')
print(f"Count after removing country-groups : {only_countries.count()}")

# select & rename fields
countries_cleared = only_countries.select(only_countries["id"].alias('iso_code'), only_countries['region']['value'].alias('region'))


Count before de-duplication : 296
Count after de-duplication : 296
Count after removing country-groups : 217


In [0]:
########################## Pollution cleanup
pollution_table = 'pollution_per_country_pipeline_project.bronze.pollution'
pollution = spark.read.table(pollution_table)

# remove empty values in required fields &&& remove dupes
print(f"Rows before clean-up : {pollution.count()}")
pollution_unique = pollution.dropna(subset=['iso_code', 'year', 'co2','population'])
pollution_unique = pollution_unique.filter(pollution_unique['co2'] > 0)
pollution_unique = pollution_unique.drop_duplicates(['iso_code','year'])
print(f"Rows after clean-up : {pollution_unique.count()}")

# select fields
pollution_cleared = pollution_unique.select(
    pollution_unique['iso_code'],
    pollution_unique['country'],
    pollution_unique['year'],
    pollution_unique['co2'],
    pollution_unique['population']
    )


Rows before clean-up : 50411
Rows after clean-up : 22386


In [0]:
from pyspark.sql.functions import coalesce, concat, lit

# join tables on iso_code
pollution_x_countries = pollution_cleared.join(countries_cleared, on='iso_code', how='left')
print(pollution_x_countries.count())

# check empty region values 
#pollution_x_countries.filter(pollution_x_countries['region'].isNull()).display()

# replace empty region values with Unclassified Region - Country
final_matrix = pollution_x_countries.withColumn(
    'region',
    coalesce(pollution_x_countries['region'],
            concat(lit('Unclassified Region - '),
                pollution_x_countries['country']))
)

final_matrix.filter(final_matrix['region'].isNull()).display()

22386


iso_code,country,year,co2,population,region


In [0]:
#save data to table in silver layer
final_matrix.writeTo('pollution_per_country_pipeline_project.silver.countries_x_pollution').createOrReplace()